# 01 — Setup and baseline generation (Tasks 1–2)

Accelerator: **GPU T4 x2**, Internet **on**. Clones the repo, installs dependencies without touching PyTorch,
checks the GPUs, runs the adapter tests, and generates **Qwen2.5-0.5B base** responses on the fixed
AlpacaEval subset (300 prompts, seed 42) with greedy decoding.

In [ ]:
# Clone (or update) the project repo into /kaggle/working. Notebook settings: Internet ON.
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")

In [ ]:
# Manual section 5.2: keep Kaggle's PyTorch; do not reinstall it.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
# Manual section 7: confirm two T4 GPUs and print package versions.
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
sh("PYTHONPATH=. python -m pytest -q tests/")

### Decoding settings
Greedy decoding. Run the sanity check first; if the base/SFT outputs loop, raise the repetition penalty
(e.g. 1.1) **here and in notebook 06**, and document it in the report.

In [ ]:
DECODE = "--max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0"  # keep identical across notebooks 01 and 06

In [ ]:
# Sanity check on 3 prompts (manual section 12): inspect prompt format, EOS, and length.
sh("python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base "
   f"--eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/sanity_base.json --limit 3 {DECODE}")

In [ ]:
sh("python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base "
   f"--eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/alpaca_base.json {DECODE}")

In [ ]:
import json
gens = json.load(open("outputs/generations/alpaca_base.json"))
print(len(gens), "generations;", sum(g["hit_max_new_tokens"] for g in gens), "hit max_new_tokens")
print(json.dumps(gens[0], indent=2)[:1500])

Now **Save Version → Save & Run All (Commit)** so `outputs/generations/` is preserved.